In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/yelp_sample_1000.csv")

print(df.shape)
print(df["stars"].value_counts().sort_index())

(1000, 3)
stars
1    200
2    200
3    200
4    200
5    200
Name: count, dtype: int64


In [23]:
from sklearn.model_selection import train_test_split

corpus_df, query_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["stars"],
    random_state=42
)


print("Corpus:\n" ,corpus_df["stars"].value_counts() )
print("Query:\n" , query_df["stars"].value_counts())

Corpus:
 stars
2    160
4    160
5    160
3    160
1    160
Name: count, dtype: int64
Query:
 stars
3    40
1    40
2    40
4    40
5    40
Name: count, dtype: int64


In [28]:
print("Common Indices : " , len(set(corpus_df.index).intersection(set(query_df.index))))

print("Common Texts : " ,len(set(corpus_df["text"]).intersection(set(query_df["text"]))))

Common Indices :  0
Common Texts :  0


In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer


vectorizer = TfidfVectorizer(
    max_features=5000 , 
    ngram_range=(1,2) , 
    lowercase=True
)


corpus_tfidf = vectorizer.fit_transform(corpus_df["text"])
query_tfidf = vectorizer.transform(query_df["text"])

print("Corpus Shape : " , corpus_tfidf.shape)
print("Query Shape : " , query_tfidf.shape)
print("Vocabulary Size : " , len(vectorizer.vocabulary_))


Corpus Shape :  (800, 5000)
Query Shape :  (200, 5000)
Vocabulary Size :  5000


In [33]:
print(type(corpus_tfidf))

print("Non-zero features:", corpus_tfidf[0].nnz)

<class 'scipy.sparse._csr.csr_matrix'>
Non-zero features: 112


In [34]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


/home/pasali/anaconda3/envs/myenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17195.24it/s]


In [35]:
sentences = [
    "The food was delicious.",
    "The meal tasted amazing."
]


embeddings = embedding_model.encode(sentences)

print(embeddings.shape)

(2, 384)


In [36]:
corpus_embeddings = embedding_model.encode(
    corpus_df["text"].tolist(),
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(corpus_embeddings.shape)

Batches: 100%|██████████| 25/25 [00:00<00:00, 35.39it/s]

(800, 384)


In [37]:
query_embeddings = embedding_model.encode(
    query_df["text"].tolist(),
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(query_embeddings.shape)

Batches: 100%|██████████| 7/7 [00:00<00:00, 35.66it/s]

(200, 384)


In [43]:
corpus_finite = np.isfinite(corpus_embeddings).all()
query_finite = np.isfinite(query_embeddings).all()

print("Corpus finite:", corpus_finite)
print("Query finite:", query_finite)

Corpus finite: True
Query finite: True


In [45]:
first_embedding = corpus_embeddings[0]

print("İlk 10 değer:", first_embedding[:10])
print("Değer sayısı:", len(first_embedding[:10]))

İlk 10 değer: [ 0.09362602  0.0404241   0.05059813 -0.0647098  -0.0085778   0.02211925
 -0.0253095  -0.04824145 -0.02224654 -0.05161175]
Değer sayısı: 10


In [46]:
l2_norm = np.linalg.norm(first_embedding)

print("L2 Norm:", l2_norm)

L2 Norm: 1.0


In [48]:
from sklearn.metrics.pairwise import cosine_similarity

tfidf_similarity = cosine_similarity(
    query_tfidf,
    corpus_tfidf
)

print(tfidf_similarity.shape)

(200, 800)


In [49]:
embedding_similarity = query_embeddings @ corpus_embeddings.T

print(embedding_similarity.shape)

(200, 800)


In [51]:
query_score = tfidf_similarity[0]

print("En yüksek benzerlik skoru:", query_score.max())

En yüksek benzerlik skoru: 0.2369097709996403


In [52]:
top1_index = query_score.argmax()

print("En benzer Corpus indeksi:", top1_index)

En benzer Corpus indeksi: 246


In [53]:
top5_indices = np.argsort(query_score)[::-1][:5]

print("Top-5 Corpus indeksleri:", top5_indices)

Top-5 Corpus indeksleri: [246 792 234  77 403]


In [55]:
print("QUERY:")
print("Yorum:", query_df.iloc[0]["text"])
print("Yıldız:", query_df.iloc[0]["stars"])

print("\nTOP-5 TF-IDF SONUÇLARI:")

for rank, idx in enumerate(top5_indices, start=1):
    print(f"\n{rank}. Corpus indeksi: {idx}")
    print("Yorum:", corpus_df.iloc[idx]["text"])
    print("Yıldız:", corpus_df.iloc[idx]["stars"])
    print("Benzerlik skoru:", query_score[idx])

QUERY:
Yorum: I am only judging this place as a bar, never had the food.  I'd rather give it 2.5 stars.\n\nI came here with a colleague after a long, long travel experience to get to Pittsburgh.  \n\nThey had a nice collection of beers including some local ones.  The only negative side is that the bartender had almost no knowlege of what the beers were like so we had to just try our luck.  The bartender had never tried any of the beers nor did he care about them.  I would have liked to have more info in order to make smarter choices about which beers to get.  At least one of them I would have avoided if I'd know more about it.\n\nPerhaps he was just not used to encountering a beer geek.\n\nI did pass by the place the next day around lunch and it was full of customers.  For what ever that means.
Yıldız: 3

TOP-5 TF-IDF SONUÇLARI:

1. Corpus indeksi: 246
Yorum: I was at the Visulite Saturday night and it was a terrible experience. I bought tickets online 2 days prior to the show. It was 

In [56]:
embedding_scores = embedding_similarity[0]

embedding_top5_indices = np.argsort(embedding_scores)[::-1][:5]

print("Embedding Top-5 indeksleri:", embedding_top5_indices)

Embedding Top-5 indeksleri: [385 466 495 350 687]


In [57]:
print("QUERY:")
print("Yorum:", query_df.iloc[0]["text"])
print("Yıldız:", query_df.iloc[0]["stars"])

print("\nTOP-5 EMBEDDING SONUÇLARI:")

for rank, idx in enumerate(embedding_top5_indices, start=1):
    print(f"\n{rank}. Corpus indeksi: {idx}")
    print("Yorum:", corpus_df.iloc[idx]["text"])
    print("Yıldız:", corpus_df.iloc[idx]["stars"])
    print("Benzerlik skoru:", embedding_scores[idx])

QUERY:
Yorum: I am only judging this place as a bar, never had the food.  I'd rather give it 2.5 stars.\n\nI came here with a colleague after a long, long travel experience to get to Pittsburgh.  \n\nThey had a nice collection of beers including some local ones.  The only negative side is that the bartender had almost no knowlege of what the beers were like so we had to just try our luck.  The bartender had never tried any of the beers nor did he care about them.  I would have liked to have more info in order to make smarter choices about which beers to get.  At least one of them I would have avoided if I'd know more about it.\n\nPerhaps he was just not used to encountering a beer geek.\n\nI did pass by the place the next day around lunch and it was full of customers.  For what ever that means.
Yıldız: 3

TOP-5 EMBEDDING SONUÇLARI:

1. Corpus indeksi: 385
Yorum: D's has been hovering among my Pittsburgh bookmarks for a couple of years now, so I was happy to finally stop by for a visit 

In [58]:
tfidf_top1_indices = np.argmax(tfidf_similarity, axis=1)
embedding_top1_indices = np.argmax(embedding_similarity, axis=1)

comparison_df = pd.DataFrame({
    "Query ID": range(len(query_df)),
    "Query Stars": query_df["stars"].to_numpy(),
    "TF-IDF Top-1 Stars": corpus_df.iloc[tfidf_top1_indices]["stars"].to_numpy(),
    "Embedding Top-1 Stars": corpus_df.iloc[embedding_top1_indices]["stars"].to_numpy()
})

print(comparison_df.shape)
display(comparison_df.head(10))

(200, 4)


,Query ID,Query Stars,TF-IDF Top-1 Stars,Embedding Top-1 Stars
0,0,3,1,4
1,1,1,2,1
2,2,2,2,1
3,3,2,3,4
4,4,4,5,5
5,5,1,1,1
6,6,3,4,1
7,7,4,3,3
8,8,3,3,5
9,9,5,4,5


In [59]:
tfidf_top1_rate = (
    comparison_df["Query Stars"] ==
    comparison_df["TF-IDF Top-1 Stars"]
).mean()

print("TF-IDF Top-1 Same-Star Rate:", tfidf_top1_rate)

TF-IDF Top-1 Same-Star Rate: 0.335


In [61]:
embedding_top1_rate = (
    comparison_df["Query Stars"] ==
    comparison_df["Embedding Top-1 Stars"]
).mean()

print("Embedding Top-1 Stars Rate:", embedding_top1_rate)

Embedding Top-1 Stars Rate: 0.305


In [62]:
tfidf_top5_indices = np.argsort(tfidf_similarity, axis=1)[:, ::-1][:, :5]

print(tfidf_top5_indices.shape)

(200, 5)


In [68]:
tfidf_top5_stars = corpus_df.iloc[
    tfidf_top5_indices.flatten()
]["stars"].to_numpy().reshape(200, 5)

print(tfidf_top5_stars.shape)
print(tfidf_top5_stars[0])

query_stars = query_df["stars"].to_numpy()

tfidf_top5_matches = (
    tfidf_top5_stars == query_stars[:, None]
)

print(tfidf_top5_matches[0])


tfidf_top5_fraction_per_query = tfidf_top5_matches.mean(axis=1)


tfidf_top5_fraction = tfidf_top5_fraction_per_query.mean()

print("TF-IDF Top-5 Same-Star Fraction:", tfidf_top5_fraction)

(200, 5)
[1 1 1 1 2]
[False False False False False]
TF-IDF Top-5 Same-Star Fraction: 0.29100000000000004


In [69]:
embedding_top5_indices = np.argsort(embedding_similarity, axis=1)[:, ::-1][:, :5]

print(embedding_top5_indices.shape)

(200, 5)


In [71]:
embedding_top5_stars = corpus_df.iloc[
    embedding_top5_indices.flatten()
]["stars"].to_numpy().reshape(200, 5)

print(embedding_top5_stars.shape)
print(embedding_top5_stars[0])

query_stars = query_df["stars"].to_numpy()

embedding_top5_matches = (
    embedding_top5_stars == query_stars[:, None]
)

print(embedding_top5_matches[0])


embedding_top5_fraction_per_query = embedding_top5_matches.mean(axis=1)


embedding_top5_fraction = embedding_top5_fraction_per_query.mean()

print("Embedding Top-5 Same-Star Fraction:", embedding_top5_fraction)

(200, 5)
[4 2 3 3 1]
[False False  True  True False]
Embedding Top-5 Same-Star Fraction: 0.29700000000000004


In [72]:
results_df = pd.DataFrame({
    "Metric": [
        "Corpus Samples",
        "Query Samples",
        "Top-1 Same-Star Rate",
        "Top-5 Same-Star Fraction"
    ],
    "TF-IDF": [
        len(corpus_df),
        len(query_df),
        tfidf_top1_rate,
        tfidf_top5_fraction
    ],
    "Embedding": [
        len(corpus_df),
        len(query_df),
        embedding_top1_rate,
        embedding_top5_fraction
    ]
})

display(results_df)

,Metric,TF-IDF,Embedding
0,Corpus Samples,800.000,800.000
1,Query Samples,200.000,200.000
2,Top-1 Same-Star Rate,0.335,0.305
3,Top-5 Same-Star Fraction,0.291,0.297


In [73]:
#Farklı sonuç veren 5 Query bul

different_queries = np.where(
    tfidf_top1_indices != embedding_top1_indices
)[0]

selected_queries = different_queries[:5]

print("Seçilen Query ID'leri:", selected_queries)

Seçilen Query ID'leri: [0 1 2 3 6]


In [74]:
for q_idx in selected_queries:

    tfidf_idx = tfidf_top1_indices[q_idx]
    embedding_idx = embedding_top1_indices[q_idx]

    print("=" * 60)
    print("QUERY ID:", q_idx)

    print("\nQUERY:")
    print("Yorum:", query_df.iloc[q_idx]["text"])
    print("Yıldız:", query_df.iloc[q_idx]["stars"])

    print("\nTF-IDF TOP-1:")
    print("Corpus ID:", tfidf_idx)
    print("Yorum:", corpus_df.iloc[tfidf_idx]["text"])
    print("Yıldız:", corpus_df.iloc[tfidf_idx]["stars"])

    print("\nEMBEDDING TOP-1:")
    print("Corpus ID:", embedding_idx)
    print("Yorum:", corpus_df.iloc[embedding_idx]["text"])
    print("Yıldız:", corpus_df.iloc[embedding_idx]["stars"])

QUERY ID: 0

QUERY:
Yorum: I am only judging this place as a bar, never had the food.  I'd rather give it 2.5 stars.\n\nI came here with a colleague after a long, long travel experience to get to Pittsburgh.  \n\nThey had a nice collection of beers including some local ones.  The only negative side is that the bartender had almost no knowlege of what the beers were like so we had to just try our luck.  The bartender had never tried any of the beers nor did he care about them.  I would have liked to have more info in order to make smarter choices about which beers to get.  At least one of them I would have avoided if I'd know more about it.\n\nPerhaps he was just not used to encountering a beer geek.\n\nI did pass by the place the next day around lunch and it was full of customers.  For what ever that means.
Yıldız: 3

TF-IDF TOP-1:
Corpus ID: 246
Yorum: I was at the Visulite Saturday night and it was a terrible experience. I bought tickets online 2 days prior to the show. It was listed

In [77]:
observation_df = pd.DataFrame({
    "Query ID": selected_queries,
    "TF-IDF Top-1 ID": tfidf_top1_indices[selected_queries],
    "Embedding Top-1 ID": embedding_top1_indices[selected_queries],
    "Gözlem": [""] * len(selected_queries)
})
observation_df["Gözlem"] = [
    "Embedding: Bira ve bar deneyimi açısından daha ilişkili.",
    "Embedding: Fast-food ve kötü hizmet açısından daha ilişkili.",
    "TF-IDF: Karşılama ve oturma sorunları açısından daha ilişkili.",
    "TF-IDF: Beklentileri karşılamayan restoran deneyimi açısından daha ilişkili.",
    "Embedding: Kahve siparişi ve barista iletişimi açısından daha ilişkili."
]

display(observation_df)

,Query ID,TF-IDF Top-1 ID,Embedding Top-1 ID,Gözlem
0,0,246,385,Embedding: Bira ve bar deneyimi açısından daha...
1,1,680,791,Embedding: Fast-food ve kötü hizmet açısından ...
2,2,742,744,TF-IDF: Karşılama ve oturma sorunları açısında...
3,3,716,450,TF-IDF: Beklentileri karşılamayan restoran den...
4,6,749,332,Embedding: Kahve siparişi ve barista iletişimi...


# Bulguların Yorumlanması

### 1. Sparse vs Dense
TF-IDF kelimelerin metinlerde ne kadar geçtiğine ve önemine bakar. Genellikle çok sayıda sıfır içeren sparse vektörler oluşturur. Embedding ise kelimelerin ve cümlelerin anlamsal ilişkilerini kullanır. Daha küçük ve dense vektörler üretir.

### 2. Benzerlik
Evet, mümkün. Çünkü embedding sadece aynı kelimelere bakmaz, cümlelerin anlamlarını da dikkate alır. Farklı kelimelerle aynı şeyi anlatan iki yorum birbirine yakın olabilir.

### 3. Retrieval
TF-IDF ortak kelimelere, embedding ise anlamsal ilişkilere odaklandığı için farklı sonuçlar bulabilir. Yaptığımız çalışmada da bazı yorumlarda embedding, bazılarında TF-IDF daha iyi sonuç verdi.

### 4. Yıldız Etiketleri
Çünkü yıldız puanı yorumun konusunu değil, genel memnuniyeti gösterir. Örneğin iki kişi 1 yıldız vermiş olabilir ama biri yemeklerden, diğeri çalışanlardan şikâyet ediyor olabilir.

### 5. Uzun Yorumlar
Modelin 256 token sınırı olduğu için uzun yorumların bir kısmı kesilebilir. Önemli bilgiler kesilen bölümde kalırsa model yorumun anlamını tam olarak yakalayamaz. Bu da yanlış veya daha az ilgili sonuçlar getirmesine neden olabilir.

### 6. Retrieval vs Classification
Retrieval, bir yoruma en çok benzeyen yorumları bulur. Classification ise yorumun hangi sınıfa ait olduğunu tahmin eder. Biz retrieval ile benzer yorumları bulduk. Classification yapsaydık yorumun kaç yıldız olduğunu tahmin etmeye çalışacaktık.

---

## Genel Değerlendirme

| Metrik | TF-IDF | Embedding |
|---|---:|---:|
| Top-1 Same-Star Rate | 0.335 | 0.305 |
| Top-5 Same-Star Fraction | 0.291 | 0.297 |

TF-IDF, Top-1 sonucunda biraz daha iyi çıktı. Top-5'te ise embedding az farkla öndeydi.

Elle incelediğimiz 5 yorumun 3'ünde embedding, 2'sinde TF-IDF daha ilişkili sonuçlar buldu.

Bence bu çalışmada en önemli öğrendiğim şey, **aynı yıldız puanına sahip yorumların mutlaka anlamsal olarak benzer olmadığı.** Ayrıca embedding her zaman TF-IDF'den daha iyi sonuç vermiyor. Hangi yöntemin daha başarılı olduğu, neyi ölçtüğümüze de bağlı.

In [78]:
def test_corpus_query_split():

    assert len(corpus_df) == 800
    assert len(query_df) == 200

  
    if "original_index" in corpus_df.columns and "original_index" in query_df.columns:
        corpus_ids = pd.Index(corpus_df["original_index"])
        query_ids = pd.Index(query_df["original_index"])
    else:
        corpus_ids = corpus_df.index
        query_ids = query_df.index

    assert corpus_ids.is_unique
    assert query_ids.is_unique
    assert corpus_ids.intersection(query_ids).empty, (
        "Satır indeksleri çakışıyor. "
        "Eğer indeksler sıfırlandıysa orijinal indeksleri korumalısın."
    )


    corpus_texts = set(corpus_df["text"].tolist())
    query_texts = set(query_df["text"].tolist())

    assert corpus_texts.isdisjoint(query_texts)


# TEST 2 — Representation Shapes
def test_representation_shapes():

    assert tfidf_similarity.shape == (200, 800)
    assert embedding_similarity.shape == (200, 800)

    assert corpus_embeddings.shape == (800, 384)
    assert query_embeddings.shape == (200, 384)

    assert np.isfinite(tfidf_similarity).all()
    assert np.isfinite(embedding_similarity).all()

    assert np.isfinite(corpus_embeddings).all()
    assert np.isfinite(query_embeddings).all()


# TEST 3 — Top-5 Retrieval
def test_top5_retrieval():

    methods = [
        (tfidf_top5_indices, tfidf_similarity),
        (embedding_top5_indices, embedding_similarity)
    ]

    for indices, similarity in methods:

      
        assert indices.shape == (200, 5)

     
        assert np.issubdtype(indices.dtype, np.integer)
        assert np.all(indices >= 0)
        assert np.all(indices < 800)

      
        for row in indices:
            assert len(np.unique(row)) == 5

      
        scores = np.take_along_axis(
            similarity, indices, axis=1
        )

        assert np.all(
            np.diff(scores, axis=1) <= 1e-7
        )


tests = [
    ("Test 1 — Corpus Query Split", test_corpus_query_split),
    ("Test 2 — Representation Shapes", test_representation_shapes),
    ("Test 3 — Top-5 Retrieval", test_top5_retrieval)
]

for name, test_func in tests:
    try:
        test_func()
        print(f"{name}: PASSED")
    except Exception as e:
        print(f"{name}: FAILED")
        print(f"  Hata: {e}")

Test 1 — Corpus Query Split: PASSED
Test 2 — Representation Shapes: PASSED
Test 3 — Top-5 Retrieval: PASSED
